# Name That LLM: blend de n-gramas treinado com o Open Pool

O texto de cada modelo carrega marcas de estilo: pontuação, conectivos, aberturas e fechamentos. Este notebook junta três classificadores lineares por média geométrica das probabilidades:

- hashing de n-gramas de caracteres (1 a 5) com SGD, treinado no train mais 2.500 textos por modelo do Open Pool;
- TF-IDF de caracteres e de palavras com regressão logística;
- o mesmo TF-IDF com marcas de estilo contadas por regex.

No fim, a saída é empurrada em parte para uma distribuição equilibrada entre os 18 modelos (Sinkhorn), porque a métrica é a balanced accuracy.

A validação é por prompt (GroupKFold), porque o teste só tem prompts que não aparecem no treino. Os pesos do blend saíram dessa validação. O código, os testes e o registro dos experimentos estão em github.com/juan-barross/name-that-llm.


In [ ]:
from pathlib import Path

Path("ntl").mkdir(exist_ok=True)

In [ ]:
%%writefile ntl/__init__.py


In [ ]:
%%writefile ntl/labels.py
import numpy as np

LABELS = (
    "gpt-5.5",
    "gpt-5.6-luna",
    "gpt-5.6-sol",
    "gpt-5.6-terra",
    "gpt-6-astra",
    "gemini-3.1-pro-preview",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-3.8-flash",
    "claude-fable-5",
    "claude-opus-4-6",
    "claude-opus-4-8",
    "claude-opus-5",
    "claude-sonnet-5",
    "qwen3.8-max",
    "glm-5.3",
    "glm-5.3-flash",
    "kimi-k3",
)


def decode(probabilities: np.ndarray) -> np.ndarray:
    return np.array(LABELS)[probabilities.argmax(axis=1)]


In [ ]:
%%writefile ntl/data.py
from pathlib import Path

import pandas as pd

KAGGLE_DIR = Path("/kaggle/input/competitions/name-that-llm")
LOCAL_DIR = Path(__file__).resolve().parents[2] / "data"


def data_dir() -> Path:
    for candidate in (KAGGLE_DIR, LOCAL_DIR):
        if (candidate / "train.csv").exists():
            return candidate
    raise FileNotFoundError(f"train.csv não está em {KAGGLE_DIR} nem em {LOCAL_DIR}")


def load(name: str, root: Path | None = None) -> pd.DataFrame:
    return pd.read_csv((root or data_dir()) / f"{name}.csv", keep_default_na=False, dtype={"text": str})


In [ ]:
%%writefile ntl/pool.py
from functools import cache
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from ntl.data import LOCAL_DIR

FILE = "ntl_open_pool.parquet"
KAGGLE_INPUT = Path("/kaggle/input")
RISKS = ["flag_gemini37_routing_risk", "flag_near_train_prompt", "flag_train_prompt_regen"]
COLUMNS = ["text", "model", "n_words", "match_weight", *RISKS]
WORDS = (36, 274)
SEED = 2027


def pool_path() -> Path:
    local = LOCAL_DIR / "open_pool" / FILE
    if local.exists():
        return local
    found = sorted(KAGGLE_INPUT.glob(f"**/{FILE}")) if KAGGLE_INPUT.exists() else []
    if not found:
        raise FileNotFoundError(f"{FILE} não está em {local} nem em {KAGGLE_INPUT}")
    return found[0]


def clean(pool: pd.DataFrame) -> pd.DataFrame:
    safe = ~pool[RISKS].any(axis=1)
    sized = pool["n_words"].between(*WORDS)
    return pool[safe & sized & (pool["match_weight"] > 0)]


def sample(pool: pd.DataFrame, per_label: int, seed: int = SEED) -> pd.DataFrame:
    keys = np.random.default_rng(seed).random(len(pool)) ** (1 / pool["match_weight"].to_numpy())
    ranked = pool.assign(sample_key=keys).sort_values("sample_key", ascending=False)
    return ranked.groupby("model").head(per_label).drop(columns="sample_key").reset_index(drop=True)


@cache
def load_pool(per_label: int) -> pd.DataFrame:
    pool = pq.read_table(pool_path(), columns=COLUMNS).to_pandas()
    return sample(clean(pool), per_label)[["text", "model"]]


In [ ]:
%%writefile ntl/prior.py
import numpy as np

ITERATIONS = 50


def normalize_rows(matrix: np.ndarray) -> np.ndarray:
    return matrix / matrix.sum(axis=1, keepdims=True)


def sinkhorn(probabilities: np.ndarray, iterations: int = ITERATIONS) -> np.ndarray:
    balanced = np.clip(probabilities, 1e-12, None)
    target = len(balanced) / balanced.shape[1]
    for _ in range(iterations):
        balanced = normalize_rows(balanced * (target / balanced.sum(axis=0)))
    return balanced


def soft_balance(probabilities: np.ndarray, weight: float) -> np.ndarray:
    if not weight:
        return probabilities
    mixed = (
        np.log(np.clip(probabilities, 1e-12, None)) * (1 - weight) + np.log(sinkhorn(probabilities)) * weight
    )
    return normalize_rows(np.exp(mixed - mixed.max(axis=1, keepdims=True)))


In [ ]:
%%writefile ntl/features.py
import re
from collections.abc import Iterable

import numpy as np
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

MARKS = {
    "em_dash": "—",
    "en_dash": "–",
    "semicolon": ";",
    "colon": ":",
    "parenthesis": r"\(",
    "curly_quote": "[“”‘’]",
    "straight_quote": "[\"']",
    "ellipsis": r"…|\.\.\.",
    "asterisk": r"\*",
    "hash": "#",
    "digit": r"\d",
    "exclamation": "!",
    "question": r"\?",
    "comma": ",",
    "newline": "\n",
}
STATS = ("log_words", "mean_word_length", "mean_sentence_words", "std_sentence_words", "uppercase_rate")
PATTERNS = {name: re.compile(pattern) for name, pattern in MARKS.items()}
SENTENCE = re.compile(r"(?<=[.!?])\s+")


def text_style(text: str) -> list[float]:
    words = text.split()
    sentences = [len(s.split()) for s in SENTENCE.split(text) if s.strip()] or [0]
    chars = max(len(text), 1)
    rates = [len(pattern.findall(text)) / chars for pattern in PATTERNS.values()]
    stats = [
        np.log1p(len(words)),
        np.mean([len(w) for w in words]) if words else 0.0,
        np.mean(sentences),
        np.std(sentences),
        sum(c.isupper() for c in text) / chars,
    ]
    return rates + stats


def style_matrix(texts: Iterable[str]) -> np.ndarray:
    return np.array([text_style(text) for text in texts], dtype=np.float32)


def char_tfidf() -> TfidfVectorizer:
    return TfidfVectorizer(
        analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True, lowercase=False, dtype=np.float32
    )


def word_tfidf() -> TfidfVectorizer:
    return TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True,
        token_pattern=r"\S+",
        dtype=np.float32,
    )


def char15_tfidf() -> TfidfVectorizer:
    return TfidfVectorizer(
        analyzer="char", ngram_range=(1, 5), min_df=3, sublinear_tf=True, lowercase=False, dtype=np.float32
    )


def char_hashing() -> Pipeline:
    hashing = HashingVectorizer(
        analyzer="char",
        ngram_range=(1, 5),
        n_features=2**20,
        alternate_sign=False,
        norm=None,
        lowercase=False,
    )
    return make_pipeline(hashing, TfidfTransformer(sublinear_tf=True))


def scaled_style() -> Pipeline:
    return make_pipeline(FunctionTransformer(style_matrix), StandardScaler())


BUILDERS = {
    "char": char_tfidf,
    "char15": char15_tfidf,
    "hash": char_hashing,
    "word": word_tfidf,
    "style": scaled_style,
}


def vectorizer(*parts: str) -> FeatureUnion:
    return FeatureUnion([(part, BUILDERS[part]()) for part in parts])


In [ ]:
%%writefile ntl/models.py
from dataclasses import dataclass

import pandas as pd
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.pipeline import Pipeline

from ntl.features import vectorizer
from ntl.pool import load_pool

SEED = 2027


@dataclass(frozen=True)
class Experiment:
    parts: tuple[str, ...]
    classifier: str = "logreg"
    strength: float = 3.0
    pool_per_label: int = 0
    pool_weight: float = 0.85


EXPERIMENTS = {
    "char_logreg": Experiment(("char",)),
    "char_word_logreg": Experiment(("char", "word")),
    "char_word_style_logreg": Experiment(("char", "word", "style")),
    "char15_word_logreg": Experiment(("char15", "word"), strength=10.0),
    "hash_sgd": Experiment(("hash",), "sgd", 2e-5),
    "hash_sgd_pool300": Experiment(("hash",), "sgd", 2e-5, pool_per_label=300),
    "hash_sgd_pool": Experiment(("hash",), "sgd", 2e-5, pool_per_label=2500),
}


def classifier(experiment: Experiment) -> LogisticRegression | SGDClassifier:
    if experiment.classifier == "sgd":
        return SGDClassifier(
            loss="log_loss",
            alpha=experiment.strength,
            class_weight="balanced",
            max_iter=50,
            random_state=SEED,
        )
    return LogisticRegression(C=experiment.strength, class_weight="balanced", max_iter=1000)


def build(name: str) -> Pipeline:
    experiment = EXPERIMENTS[name]
    return Pipeline([("features", vectorizer(*experiment.parts)), ("classifier", classifier(experiment))])


def training_frame(name: str, train: pd.DataFrame) -> pd.DataFrame:
    experiment = EXPERIMENTS[name]
    own = train[["text", "model"]].assign(weight=1.0)
    if not experiment.pool_per_label:
        return own
    pool = load_pool(experiment.pool_per_label).assign(weight=experiment.pool_weight)
    return pd.concat([own, pool], ignore_index=True)


def fit(name: str, train: pd.DataFrame) -> Pipeline:
    data = training_frame(name, train)
    return build(name).fit(data["text"], data["model"], classifier__sample_weight=data["weight"].to_numpy())


In [ ]:
%%writefile ntl/predict.py
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline

from ntl.blend import geometric_blend
from ntl.labels import LABELS, decode
from ntl.models import fit
from ntl.prior import soft_balance

INDEX = {label: i for i, label in enumerate(LABELS)}


def aligned_probabilities(model: Pipeline, texts: pd.Series) -> np.ndarray:
    unknown = set(model.classes_) - set(LABELS)
    if unknown:
        raise ValueError(f"rótulos fora da lista no treino: {sorted(unknown)}")
    probabilities = np.zeros((len(texts), len(LABELS)))
    probabilities[:, [INDEX[label] for label in model.classes_]] = model.predict_proba(texts)
    return probabilities


def fit_predict(
    blend: dict[str, float], train: pd.DataFrame, test: pd.DataFrame, balance: float
) -> np.ndarray:
    members = [aligned_probabilities(fit(name, train), test["text"]) for name in blend]
    return decode(soft_balance(geometric_blend(members, list(blend.values())), balance))


In [ ]:
%%writefile ntl/blend.py
from collections.abc import Sequence

import numpy as np
from sklearn.metrics import balanced_accuracy_score

from ntl.labels import decode
from ntl.prior import normalize_rows

GRID = tuple(np.round(np.linspace(0.1, 1.0, 10), 2))
MIN_GAIN = 0.005


def geometric_blend(members: Sequence[np.ndarray], weights: Sequence[float]) -> np.ndarray:
    logits = sum(w * np.log(np.clip(p, 1e-12, None)) for p, w in zip(members, weights, strict=True))
    return normalize_rows(np.exp(logits - logits.max(axis=1, keepdims=True)))


def blend_score(members: Sequence[np.ndarray], weights: Sequence[float], labels: np.ndarray) -> float:
    return balanced_accuracy_score(labels, decode(geometric_blend(members, weights)))


def search_weights(members: Sequence[np.ndarray], labels: np.ndarray) -> tuple[list[float], float]:
    singles = [blend_score([m], [1.0], labels) for m in members]
    weights = [0.0] * len(members)
    weights[int(np.argmax(singles))] = 1.0
    best = max(singles)
    improved = True
    while improved:
        improved = False
        for i in range(len(members)):
            for w in GRID:
                trial = [*weights[:i], w, *weights[i + 1 :]]
                score = blend_score(members, trial, labels)
                if score > best + MIN_GAIN:
                    weights, best, improved = trial, score, True
    return weights, best


In [ ]:
%%writefile ntl/validation.py
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold

from ntl.labels import LABELS, decode
from ntl.models import fit
from ntl.predict import aligned_probabilities

SEED = 2027
N_SPLITS = 5


@dataclass(frozen=True)
class CrossValidation:
    probabilities: np.ndarray
    fold_scores: tuple[float, ...]

    @property
    def score(self) -> float:
        return float(np.mean(self.fold_scores))


def folds(groups: pd.Series, n_splits: int = N_SPLITS) -> list[tuple[np.ndarray, np.ndarray]]:
    splitter = GroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    return list(splitter.split(groups, groups=groups))


def cross_validate(experiment: str, train: pd.DataFrame) -> CrossValidation:
    probabilities = np.zeros((len(train), len(LABELS)))
    scores = []
    for fit_index, valid_index in folds(train["prompt_id"]):
        valid = train.iloc[valid_index]
        model = fit(experiment, train.iloc[fit_index])
        probabilities[valid_index] = aligned_probabilities(model, valid["text"])
        scores.append(balanced_accuracy_score(valid["model"], decode(probabilities[valid_index])))
    return CrossValidation(probabilities, tuple(scores))


In [ ]:
%%writefile ntl/submission.py
from pathlib import Path

import pandas as pd

from ntl.labels import LABELS

COLUMNS = ["id", "model"]


def validate(submission: pd.DataFrame) -> None:
    if list(submission.columns) != COLUMNS:
        raise ValueError(f"colunas devem ser {COLUMNS}, vieram {list(submission.columns)}")
    if submission.isna().any().any():
        raise ValueError("submissão com valor nulo")
    if submission["id"].duplicated().any():
        raise ValueError("submissão com id repetido")
    unknown = set(submission["model"]) - set(LABELS)
    if unknown:
        raise ValueError(f"rótulos fora da lista: {sorted(unknown)}")


def write(ids: pd.Series, predictions, path: Path = Path("submission.parquet")) -> pd.DataFrame:
    submission = pd.DataFrame({"id": ids.to_numpy(), "model": list(predictions)})
    validate(submission)
    submission.to_parquet(path, index=False)
    return submission


In [ ]:
from ntl.data import load
from ntl.predict import fit_predict
from ntl.submission import write

BLEND = {'hash_sgd_pool': 1.0, 'char15_word_logreg': 0.4, 'char_word_style_logreg': 0.3}
BALANCE = 0.3

train, test = load("train"), load("test")
submission = write(test["id"], fit_predict(BLEND, train, test, BALANCE))
submission["model"].value_counts()